# Website Landing Page A/B Test

**Business question:** should the e-commerce team replace the old landing page with the new one?

This notebook walks through the full analysis end to end:

1. Load the raw data
2. Clean it (and check the cleaning worked)
3. Save the clean data to CSV and SQLite
4. Run the headline two-proportion z-test (control vs. treatment)
5. Break the result down by country and by day, as sanity checks
6. Write a plain-English recommendation

Every step below matches `analyze.py` in this project, just spread across cells with
commentary in between, so you can run it piece by piece and inspect the data as you go.


## 0. Setup

In [1]:
from math import erf, sqrt
from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd

try:
    from statsmodels.stats.proportion import proportions_ztest
except ImportError:
    # Fallback pooled two-proportion z-test, used only if statsmodels isn't installed.
    def proportions_ztest(count, nobs):
        successes_a, successes_b = count
        n_a, n_b = nobs
        p_pool = (successes_a + successes_b) / (n_a + n_b)
        se_pool = sqrt(p_pool * (1 - p_pool) * (1 / n_a + 1 / n_b))
        z = (successes_a / n_a - successes_b / n_b) / se_pool
        p_value = 2 * (1 - (1 + erf(abs(z) / sqrt(2))) / 2)
        return z, p_value

ROOT = Path(".").resolve()
DATA = ROOT / "data"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)


## 1. Load the raw data

Two source files, both untouched from how they were supplied:

- **ab_data.csv** — 294,478 rows: `user_id`, `timestamp`, `group` (control/treatment),
  `landing_page` (old_page/new_page), `converted` (0/1)
- **countries.csv** — user-to-country mapping (UK, US, CA)


In [2]:
ab = pd.read_csv(DATA / "ab_data.csv", parse_dates=["timestamp"])
countries = pd.read_csv(DATA / "countries.csv")

raw_rows = len(ab)
print(f"ab_data.csv: {raw_rows:,} rows")
print(f"countries.csv: {len(countries):,} rows")
ab.head()


ab_data.csv: 294,478 rows
countries.csv: 290,584 rows


,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


## 2. Clean the data

Two problems can invalidate a row:

- **Invalid assignment/page pairing.** `control` should only ever see `old_page`, and
  `treatment` only `new_page`. If a row breaks that, we can't trust which page the user
  actually saw, so it's dropped rather than guessed at.
- **Repeat exposures.** Some `user_id`s appear more than once. To keep each user as one
  independent observation, we keep only their **first valid** exposure (earliest timestamp).

Country is then attached with a left join, so any user missing from `countries.csv`
stays in the analysis with a blank country rather than being silently dropped.


In [3]:
mismatch = (
    (ab.group.eq("control") & ab.landing_page.ne("old_page"))
    | (ab.group.eq("treatment") & ab.landing_page.ne("new_page"))
)

valid = (
    ab.loc[~mismatch]
    .sort_values("timestamp")
    .drop_duplicates("user_id", keep="first")
    .copy()
)
valid = valid.merge(countries.drop_duplicates("user_id"), on="user_id", how="left")

print(f"Mismatched rows removed:        {mismatch.sum():,}")
print(f"Duplicate-user rows removed:    {(~mismatch).sum() - len(valid):,}")
print(f"Clean rows remaining:           {len(valid):,}")


Mismatched rows removed:        3,893
Duplicate-user rows removed:    1
Clean rows remaining:           290,584


**Checks** — if either of these fails, the cleaning logic is broken and nothing downstream should be trusted.

In [4]:
assert valid.user_id.duplicated().sum() == 0, "clean table still has duplicate users"
assert set(zip(valid.group, valid.landing_page)) == {
    ("control", "old_page"),
    ("treatment", "new_page"),
}, "clean table still has invalid group/page pairings"
print("Both checks passed: one row per user, only valid group/page pairings remain.")


Both checks passed: one row per user, only valid group/page pairings remain.


## 3. Save the clean data

The clean table goes to `outputs/ab_clean.csv`, and all three tables (raw, clean,
countries) go into a SQLite database so the same checks can be re-run in SQL
(see `sql/sql_queries.sql`).


In [5]:
valid.to_csv(OUT / "ab_clean.csv", index=False)

db = sqlite3.connect(OUT / "ab_test.sqlite")
ab.to_sql("ab_raw", db, if_exists="replace", index=False)
valid.to_sql("ab_clean", db, if_exists="replace", index=False)
countries.to_sql("countries", db, if_exists="replace", index=False)
db.close()
print("Saved ab_clean.csv and ab_test.sqlite (tables: ab_raw, ab_clean, countries).")


Saved ab_clean.csv and ab_test.sqlite (tables: ab_raw, ab_clean, countries).


## 4. The headline test: does the new page convert better than the old page?

- **Metric:** conversion rate (binary outcome per user)
- **Null hypothesis (H0):** control and treatment conversion rates are equal
- **Alternative (H1):** they differ
- **Test:** two-proportion z-test (appropriate for comparing two binary rates)


In [6]:
summary = (
    valid.groupby("group")
    .agg(
        users=("user_id", "nunique"),
        conversions=("converted", "sum"),
        conversion_rate=("converted", "mean"),
    )
    .reset_index()
)

by_group = summary.set_index("group")
control, treatment = by_group.loc["control"], by_group.loc["treatment"]

counts = np.array([treatment.conversions, control.conversions])
nobs = np.array([treatment.users, control.users])
z, p = proportions_ztest(counts, nobs)

diff = treatment.conversion_rate - control.conversion_rate
se = sqrt(
    treatment.conversion_rate * (1 - treatment.conversion_rate) / treatment.users
    + control.conversion_rate * (1 - control.conversion_rate) / control.users
)
ci_low, ci_high = diff - 1.96 * se, diff + 1.96 * se

summary["raw_rows"] = raw_rows
summary["mismatched_rows_removed"] = int(mismatch.sum())
summary["duplicate_user_rows_removed_after_mismatch"] = int((~mismatch).sum() - len(valid))
summary["treatment_minus_control_pp"] = diff * 100
summary["relative_change_pct"] = diff / control.conversion_rate * 100
summary["z_statistic"] = z
summary["p_value"] = p
summary["ci_95_lower_pp"] = ci_low * 100
summary["ci_95_upper_pp"] = ci_high * 100
summary.to_csv(OUT / "ab_test_summary.csv", index=False)

print(f"Control conversion:    {control.conversion_rate:.3%}")
print(f"Treatment conversion:  {treatment.conversion_rate:.3%}")
print(f"Difference:            {diff*100:+.3f} pp  ({diff/control.conversion_rate*100:+.2f}% relative)")
print(f"z = {z:.4f}, p = {p:.4f}")
print(f"95% CI on the difference: [{ci_low*100:.3f}, {ci_high*100:.3f}] pp")


Control conversion:    12.039%
Treatment conversion:  11.881%
Difference:            -0.158 pp  (-1.31% relative)
z = -1.3109, p = 0.1899
95% CI on the difference: [-0.394, 0.078] pp


**Reading this result:** the p-value (0.19) is well above the conventional 0.05
threshold, and the 95% confidence interval for the difference spans from a small
loss to a small gain (crosses zero). That means this experiment did not find
statistically significant evidence that the new page performs differently from
the old one — not that they're proven equal, just that this test can't tell them
apart with confidence.


## 5a. Country breakdown (exploratory)

Splitting by country is useful for spotting anything unbalanced or a segment that
behaves very differently — but it was not the pre-registered test, so treat it as a
sanity check, not a standalone shipping decision (and correct for multiple
comparisons before treating any single country's p-value as conclusive).


In [7]:
def country_test(group_df):
    s = group_df.groupby("group").converted.agg(["sum", "count", "mean"])
    if set(s.index) != {"control", "treatment"}:
        return pd.Series(dtype=float)
    zz, pp = proportions_ztest(
        [s.loc["treatment", "sum"], s.loc["control", "sum"]],
        [s.loc["treatment", "count"], s.loc["control", "count"]],
    )
    d = s.loc["treatment", "mean"] - s.loc["control", "mean"]
    return pd.Series(
        {
            "control_users": s.loc["control", "count"],
            "treatment_users": s.loc["treatment", "count"],
            "control_rate": s.loc["control", "mean"],
            "treatment_rate": s.loc["treatment", "mean"],
            "difference_pp": 100 * d,
            "p_value": pp,
        }
    )


country = valid.groupby("country").apply(country_test).reset_index()
country.to_csv(OUT / "country_results.csv", index=False)
country


,country,control_users,treatment_users,control_rate,treatment_rate,difference_pp,p_value
0,CA,7198.0,7301.0,0.118783,0.111902,-0.688052,0.194666
1,UK,36360.0,36106.0,0.120022,0.121171,0.114899,0.634865
2,US,101716.0,101903.0,0.120630,0.118466,-0.216439,0.132274


## 5b. Daily conversion rate (stability check)\n\nA sudden jump or drift over the test's run would be a red flag; a fairly flat, noisy line for both groups is what you want to see.

In [8]:
valid["date"] = valid.timestamp.dt.date
daily = (
    valid.groupby(["date", "group"])
    .agg(users=("user_id", "nunique"), conversion_rate=("converted", "mean"))
    .reset_index()
)
daily.to_csv(OUT / "daily_conversion.csv", index=False)
daily.head(10)


,date,group,users,conversion_rate
0,2017-01-02,control,2859,0.125568
1,2017-01-02,treatment,2853,0.119874
2,2017-01-03,control,6590,0.113809
3,2017-01-03,treatment,6618,0.113781
4,2017-01-04,control,6578,0.121922
5,2017-01-04,treatment,6541,0.116649
6,2017-01-05,control,6427,0.123230
7,2017-01-05,treatment,6505,0.114988
8,2017-01-06,control,6606,0.115350
9,2017-01-06,treatment,6747,0.123462


## 6. Data-quality audit and business recommendation

In [9]:
audit = pd.DataFrame(
    [
        {
            "raw_rows": raw_rows,
            "mismatched_rows": int(mismatch.sum()),
            "raw_duplicate_user_ids": int(ab.user_id.duplicated().sum()),
            "clean_rows": len(valid),
            "clean_duplicate_user_ids": int(valid.user_id.duplicated().sum()),
            "start_date": valid.timestamp.min(),
            "end_date": valid.timestamp.max(),
            "test_days": (valid.timestamp.max() - valid.timestamp.min()).days + 1,
        }
    ]
)
audit.to_csv(OUT / "data_quality_audit.csv", index=False)
audit


,raw_rows,mismatched_rows,raw_duplicate_user_ids,clean_rows,clean_duplicate_user_ids,start_date,end_date,test_days
0,294478,3893,3894,290584,0,2017-01-02 13:42:05.378582,2017-01-24 13:41:54.460509,22


In [10]:
monthly_traffic = 100_000  # illustrative, not the site's actual traffic

business = pd.DataFrame(
    [
        {
            "assumed_monthly_traffic": monthly_traffic,
            "observed_difference_pp": diff * 100,
            "estimated_monthly_incremental_conversions": diff * monthly_traffic,
            "recommendation": (
                "Do not ship based on this test alone; p-value exceeds 0.05 "
                "and the 95% CI includes both a modest loss and a modest gain."
            ),
        }
    ]
)
business.to_csv(OUT / "business_recommendation.csv", index=False)
business


,assumed_monthly_traffic,observed_difference_pp,estimated_monthly_incremental_conversions,recommendation
0,100000,-0.157824,-157.823899,Do not ship based on this test alone; p-value ...


## Conclusion

**Do not ship the new page based on this experiment alone.** The point estimate is
slightly negative (new page converts about 0.16 percentage points worse), but the
p-value (0.19) and confidence interval both say the test can't distinguish this from
no effect at all. At an illustrative 100,000 monthly visits, the observed effect
would be roughly 158 fewer conversions — but the true effect could just as easily be
a small gain.

The test ran for 22 days. Country and daily results look reasonably balanced and
stable, supporting that the randomization worked as intended. A follow-up test
should be pre-powered around a specific minimum lift worth detecting, since this
one's confidence interval is too wide to rule out a meaningful gain or loss.

**Limitations:** the dataset has no revenue, margin, device, or traffic-source
information, and no pre-registered minimum-detectable-effect. Country cuts are
exploratory and should be corrected for multiple comparisons before being used to
justify anything on their own.
